# Testar Base de Validação 

Este notebook tem a função do avaliador poder verificar o **modelo final** com sua base de dados secreta


**Pipeline:**
1. Carregar a base bruta (CSV)
2. Aplicar o pipeline de tratamento (`pipeline_tratamento`)
3. Avaliar o CatBoost com validação cruzada
4. Interpretar os resultados

## 1. Importação de Bibliotecas

In [ ]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from utils import pipeline_tratamento, avaliar_modelo

## 2. Carregamento da Base de Competição

> **Instruções:** Carregue seu database na variável `df_competicao`.
> O DataFrame deve conter as mesmas 34 colunas do dataset original de treinamento

In [ ]:
# --> Escreva o df_competicao_aqui:


# df_competicao = pd.read_csv('base_competicao.csv')


## 3. Aplicação do Pipeline de Tratamento

O `pipeline_tratamento` aplica automaticamente, nesta ordem

Todas essas funções foram definidas e explicadas no arquivo de **Tratamento**

| # | Etapa | Descrição |
|---|-------|-----------|
| 1 | `tratar_variaveis_categoricas` | Label encoding, one-hot encoding e normalização de resolução |
| 2 | `aplicar_feature_engineering` | Features de sinergia (power_index, ram_per_core, etc.) |
| 3 | `correcao_feature` | Features de correção de viés (Apple Tax, depreciação temporal) |
| 4 | `normalizar_variaveis_numericas` | MinMaxScaler nas variáveis contínuas |
| 5 | `remover_colunas_colineares` | Remoção de features redundantes |

In [ ]:
df_tratado = pipeline_tratamento(df_competicao)

X_tree = df_tratado.drop(columns=['price_log'])
y_tree = df_tratado['price_log']

print(f"\nX_tree: {X_tree.shape}")

## 4. Avaliação do CatBoost

Hiperparâmetros otimizados via Optuna no notebook de modelagem.
Avaliação com **validação cruzada K-Fold (k=5)** e transformação inversa (`expm1`) para métricas em dólares.

In [ ]:
k_fold = 5

catboost_modelo = CatBoostRegressor(
    iterations=849,
    learning_rate=0.0593,
    depth=5,
    l2_leaf_reg=9.819,
    subsample=0.720,
    bootstrap_type='Bernoulli',
    random_seed=42,
    verbose=False,
    thread_count=-1
)

print(">>> AVALIANDO CATBOOST NA BASE DE COMPETIÇÃO <<<\n")

avaliar_modelo(
    modelo=catboost_modelo,
    X=X_tree,
    y=y_tree,
    nome_modelo="CatBoost Otimizado",
    cv=k_fold,
    transformacao="log1p"
)